# Insights Lab: Model Training

Train Logistic Regression and Random Forest models, then export the best pipeline and metadata.

In [ ]:
from pathlib import Path

import json
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ARTIFACTS_DIR = Path('..') / 'artifacts'
cleaned_path = ARTIFACTS_DIR / 'cleaned_dataset.csv'
df = pd.read_csv(cleaned_path)

feature_cols = ['cgpa', 'internships', 'work_experience', 'branch', 'gender']
target_col = 'placed'

df['work_experience'] = df.get('work_experience', df['internships']).fillna(df['internships'])

X = df[feature_cols]
y = df[target_col]

numeric_features = ['cgpa', 'internships', 'work_experience']
categorical_features = ['branch', 'gender']

preprocessor = ColumnTransformer(
    [("num", StandardScaler(), numeric_features),
     ("cat", OneHotEncoder(handle_unknown='ignore'), categorical_features)]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
log_reg = Pipeline([
preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=500))
])

rf = Pipeline([
preprocess", preprocessor),
    ("model", RandomForestClassifier(n_estimators=250, random_state=42))
])

models = {"logistic_regression": log_reg, "random_forest": rf}
scores = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    scores[name] = {
        "accuracy": accuracy_score(y_test, preds),
        "roc_auc": roc_auc_score(y_test, proba)
    }

scores

In [ ]:
best_model_name = max(scores, key=lambda key: scores[key]['accuracy'])
best_model = models[best_model_name]
best_model.fit(X_train, y_train)

pipeline_path = ARTIFACTS_DIR / 'placement_pipeline.joblib'
joblib.dump(best_model, pipeline_path)

pipeline_path

In [ ]:
preprocessor = best_model.named_steps['preprocess']
model = best_model.named_steps['model']

feature_names = preprocessor.get_feature_names_out()

if hasattr(model, 'feature_importances_'):
    importances = model.feature_importances_
elif hasattr(model, 'coef_'):
    importances = np.abs(model.coef_[0])
else:
    importances = np.zeros(len(feature_names))

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)
importance_df['importance'] = importance_df['importance'].astype(float)

importance_df.head(10)

In [ ]:
processed_X = preprocessor.transform(X)
if hasattr(processed_X, 'toarray'):
    processed_X = processed_X.toarray()

processed_df = pd.DataFrame(processed_X, columns=feature_names)
processed_df['placed'] = y.values

processed_path = ARTIFACTS_DIR / 'processed_dataset.csv'
processed_df.to_csv(processed_path, index=False)

processed_path

In [ ]:
metadata = {
    'model_used': best_model_name,
    'accuracy': round(scores[best_model_name]['accuracy'], 4),
    'roc_auc': round(scores[best_model_name]['roc_auc'], 4),
    'raw_feature_columns': feature_cols,
    'numeric_features': numeric_features,
    'categorical_features': categorical_features,
    'top_features': importance_df.head(8).to_dict(orient='records')
}

meta_path = ARTIFACTS_DIR / 'insights_metadata.json'
meta_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')

meta_path